# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access and print basic metadata
md = dataset.metadata
print(f"Name: {md.name}")
print(f"Description: {md.description}")
print(f"Keywords: {getattr(md, 'keywords', '')}")
print(f"License: {getattr(md, 'license', '')}")
print(f"Published: {getattr(md, 'datePublished', '')}")

## 2. Data Overview
Review available record sets, fields, their `@id` values, and the data schema.

Explore all record set entities defined in the dataset schema.

In [ ]:
# List available record sets and their fields using the mlcroissant metadata API
record_set_ids = [rs['@id'] for rs in getattr(md, 'recordSet', [])]
print(f"Available record sets: {record_set_ids if record_set_ids else '[None found. See the instructions below.]'}")

# If record sets are not present at the top-level, try to extract them from the metadata's parts/hasPart
if not record_set_ids:
    # Try to recursively search for `@type` == 'RecordSet' in hasPart
    def find_record_sets(obj):
        found = []
        if isinstance(obj, dict):
            if obj.get('@type') == 'RecordSet' or obj.get('@type') == 'cr:RecordSet':
                found.append(obj)
            for v in obj.values():
                found.extend(find_record_sets(v))
        elif isinstance(obj, list):
            for v in obj:
                found.extend(find_record_sets(v))
        return found

    all_md_json = md.to_json() if hasattr(md, 'to_json') else md
    rec_sets = find_record_sets(all_md_json)
    record_set_ids = [rec['@id'] for rec in rec_sets if '@id' in rec]
    print(f"Discovered record sets (by recursive search): {record_set_ids}")

# Show example schema for fields in the first record set
if record_set_ids:
    rs_id = record_set_ids[0]
    print(f"\nFields and columns for record set: {rs_id}")
    # Find the record set definition
    rec_set_obj = None
    if hasattr(md, 'recordSet') and isinstance(md.recordSet, list):
        for rs_obj in md.recordSet:
            if rs_obj['@id'] == rs_id:
                rec_set_obj = rs_obj
    if not rec_set_obj:
        # Fallback: from our recursive find
        for rec in find_record_sets(all_md_json):
            if rec['@id'] == rs_id:
                rec_set_obj = rec
    # Get fields info
    field_ids = []
    if 'field' in rec_set_obj:
        if isinstance(rec_set_obj['field'], list):
            for f in rec_set_obj['field']:
                if isinstance(f, dict) and '@id' in f:
                    field_ids.append(f['@id'])
                elif isinstance(f, str):
                    field_ids.append(f)
        else:
            field_ids.append(rec_set_obj['field']['@id'] if isinstance(rec_set_obj['field'], dict) else rec_set_obj['field'])
    print(f"Field @ids: {field_ids}")
    # Detailed field information if available
    if 'field' in rec_set_obj and isinstance(rec_set_obj['field'], list):
        print("Example field definitions:")
        for field in rec_set_obj['field'][:3]:  # Show up to 3 fields
            print(field)
else:
    print("No record sets found in this dataset.")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We use the `@id` of the record set to extract records and explore their field names.

In [ ]:
# If you have identified available record set ids, use them here,
# otherwise use the example: 'https://api.app.sen.science/frontiers/7154140/recset1' (Replace with correct one if needed)

if record_set_ids:
    print(f"Extracting records for record sets: {record_set_ids}")
    dataframes = {}
    for recset_id in record_set_ids:
        records = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(records)
        dataframes[recset_id] = df
        print(f"\nRecord set: {recset_id}")
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
else:
    print("No available record set IDs to extract records.")

## 4. Exploratory Data Analysis (EDA)
Let's select one numeric field and perform filtering and normalization.

We'll demonstrate with the first record set and its first numeric field (replace field id as appropriate after inspecting columns above).

In [ ]:
import numpy as np

# For demonstration purposes, pick the first record set and choose a likely numeric field
if record_set_ids:
    recset_id = record_set_ids[0]
    df = dataframes[recset_id]
    # Try to locate a numeric field by inspecting column names and types
    numeric_field = None
    for col in df.columns:
        col_data = df[col].dropna()
        sample = col_data.iloc[0] if not col_data.empty else None
        # Heuristically, select the first column with a float or int type
        try:
            # Try to cast sample to float
            val = float(sample) if sample is not None else None
            if val is not None:
                numeric_field = col
                break
        except Exception:
            continue
    if numeric_field:
        print(f"Using numeric field: {numeric_field}")
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean: {threshold:.3f}")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print("Normalized values:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field (not the numeric field)
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].nunique() > 1 and df[col].nunique() < 20:
                group_field = col
                break
        if group_field:
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found.")
    else:
        print("No numeric field found to filter/normalize.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or field relationships for your dataset.

Let's plot a histogram and pairplot to visualize relationships if numeric fields are available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids:
    recset_id = record_set_ids[0]
    df = dataframes[recset_id]
    # Try to plot histogram for a numeric field
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if not numeric_cols:
        # fallback: try to cast columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                continue
        numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_cols:
        col = numeric_cols[0]
        plt.figure(figsize=(6, 4))
        df[col].dropna().hist(bins=30)
        plt.title(f"Distribution of {col}")
        plt.xlabel(col)
        plt.ylabel("Count")
        plt.show()
        # Pairplot if multiple numeric columns
        if len(numeric_cols) > 1:
            sns.pairplot(df[numeric_cols].dropna())
            plt.show()
    else:
        print("No numeric columns found to visualize.")
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to explore and process the FAIR^2 dataset.

Key steps included loading structured biomedical proteomics data, discovering record sets and schema via their `@id`, extracting records to DataFrames, performing basic EDA (including filtering, normalization, and grouping), and plotting visual summaries.

For advanced bioinformatics or proteomics analysis, refer to field definitions in the Croissant schema and tailor downstream workflows accordingly.